# Fine-tuning YOLO for fire and smoke (gateway vision model)

**What this notebook does, in order**
1. checks the machine and the **GPU** (stops with the exact fix if PyTorch cannot see it),
2. loads the YOLO model and its pretrained weights,
3. checks the D-Fire data and the leak-free split `split_v2` (RES-CAM-3),
4. prepares the images: lists, OV2640-like degraded copies, night-like negatives, camera-stamp randomisation,
5. fine-tunes in two phases (head first, then everything) on the GPU and exports ONNX,
6. measures what matters for the decision: **false alarms per day** at a fixed recall (with confidence intervals), day / night and camera-stamp slices,
7. chooses the thresholds on the validation split only, calibrates the scores, and writes `thresholds.json`,
8. checks the ONNX export against the PyTorch model,
9. starts the **gateway service** with the new model and **sends photos to it like a box does**, then shows the verdicts and the boxes.

**Before the first full run, set `PRESET = "smoke"` (a few minutes): it validates the machine, the paths and the whole chain on 1 % of the data.** Then use `"full"`.

Honest limits, repeated in the last cell: the model never sees a real OV2640 image unless you add some; degradations are assumptions; detector scores are not probabilities until calibrated; D-Fire sites are unknown, so site-level leakage cannot be excluded.

In [ ]:
# ---------------------------------------------------------------- CONFIG (edit here)
PRESET = "full"            # "smoke" (1 % of the data, 1 epoch: checks the chain) | "quick" (5+20 epochs) | "full" (10+60 epochs)
MODEL_WEIGHTS = "yolov8s.pt"   # yolov8n / s / m / l / x .pt  (or yolo11s.pt ...). Downloaded automatically on first use.
IMGSZ = 640                # fire boxes are small (median 0.5 % of the image): try 960 if the GPU has the memory
BATCH = None               # None = chosen from the GPU memory; or an integer
WORKERS = None             # None = 4 on Windows, 8 elsewhere
RECALL_TARGET = 0.90       # recall wanted on validation for each class when the thresholds are chosen
FRAMES_PER_DAY = 2880      # photos per day on a scene without fire (worst case: one every 30 s all day)
SKIP_IF_DONE = True        # reuse trained weights found in OUT_DIR instead of training again
ALLOW_CPU = False          # True only for the smoke test on a machine without GPU

import os, sys, json, time, shutil, platform, subprocess, importlib.util, pathlib, math, random, io
import numpy as np

def find_repo():
    env = os.environ.get("CAS_REPO")
    if env: return pathlib.Path(env).resolve()
    p = pathlib.Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "ai-data").exists(): return q
    raise FileNotFoundError("Run the notebook inside the CAS repository (or set the CAS_REPO environment variable).")

REPO   = find_repo()
VISION = REPO / "ai-data/gateway/vision"
TRAIN  = VISION / "train"
FIRE_ROOT = pathlib.Path(os.environ.get("FIRE_ROOT", REPO / "ai-data/data/fire detection"))
SPLIT_CSV = pathlib.Path(os.environ.get("SPLIT_CSV", REPO / "ai-data/data/vision_split/split_v2.csv"))
OUT_DIR   = pathlib.Path(os.environ.get("OUT_DIR", TRAIN / "runs" / (MODEL_WEIGHTS.replace(".pt", "") + ("_smoke" if PRESET == "smoke" else ""))))
MODELS_DIR = VISION / "models"
MY_PHOTOS = REPO / "ai-data/data/camera_photos"        # put real OV2640 photos here (optional)
OUT_DIR.mkdir(parents=True, exist_ok=True); MODELS_DIR.mkdir(parents=True, exist_ok=True)
sys.path[:0] = [str(TRAIN), str(VISION / "python/src")]
(TRAIN / ".gitignore").write_text("runs/\ndeg/\ndeg_smoke/\nlists/\n*.pt\n") if not (TRAIN / ".gitignore").exists() else None

PRESETS = {"smoke": dict(epochs1=1, epochs2=1, deg_limit=60,   eval_limit=150,  smoke=True),
           "quick": dict(epochs1=5, epochs2=20, deg_limit=6000, eval_limit=None, smoke=False),
           "full":  dict(epochs1=10, epochs2=60, deg_limit=0,   eval_limit=None, smoke=False)}
P = PRESETS[PRESET]
MODEL_WEIGHTS = os.environ.get("MODEL_WEIGHTS", MODEL_WEIGHTS)
print(f"repo        : {REPO}\ndata        : {FIRE_ROOT}\nsplit csv   : {SPLIT_CSV}\noutput      : {OUT_DIR}\npreset      : {PRESET} {P}")

## 1. Machine and GPU
Training on CPU would take days. The cell stops if PyTorch cannot see a CUDA GPU, and prints the fix. **On Windows, `pip install ultralytics` installs a CPU-only PyTorch**: install the CUDA build first (the command is printed), restart the kernel, then run again.

In [ ]:
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is not installed. Install the CUDA build first, for example:\n"
                       "  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121\n"
                       "(pick cu118 / cu121 / cu124 to match `nvidia-smi`), restart the kernel, run this cell again.")
import torch
print("python", platform.python_version(), "|", platform.system(), "| torch", torch.__version__, "| built for CUDA", torch.version.cuda)
if torch.cuda.is_available():
    pr = torch.cuda.get_device_properties(0)
    GPU_NAME, VRAM_GB = pr.name, pr.total_memory / 2**30
    DEVICE = "0"
    print(f"GPU 0: {GPU_NAME} | {VRAM_GB:.1f} GB | compute capability {pr.major}.{pr.minor} | {torch.cuda.device_count()} device(s)")
else:
    GPU_NAME, VRAM_GB, DEVICE = None, 0.0, "cpu"
    smi = shutil.which("nvidia-smi")
    msg = ("PyTorch cannot see a CUDA GPU.\n"
           + ("`nvidia-smi` exists, so the driver is there but this PyTorch is a CPU build. Fix:\n"
              "  pip uninstall -y torch torchvision && pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121\n"
              "then restart the kernel and run again.\n" if smi else
              "No `nvidia-smi` found: install the NVIDIA driver (and a CUDA build of PyTorch), or run on another machine.\n"))
    if not ALLOW_CPU:
        raise RuntimeError(msg + "Set ALLOW_CPU = True only to test the chain with PRESET='smoke'.")
    print("WARNING (CPU mode, smoke test only):\n" + msg)

need = [("ultralytics", "ultralytics"), ("cv2", "opencv-python"), ("yaml", "pyyaml"), ("onnx", "onnx"), ("onnxruntime", "onnxruntime"),
        ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "pillow"), ("tqdm", "tqdm")]
missing = [pip for mod, pip in need if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import ultralytics
print("ultralytics", ultralytics.__version__)

if WORKERS is None: WORKERS = 4 if platform.system() == "Windows" else 8
if BATCH is None:
    mult = {"n": 1.4, "s": 1.0, "m": 0.55, "l": 0.35, "x": 0.25}.get(MODEL_WEIGHTS.replace("yolov8", "").replace("yolo11", "")[:1], 1.0)
    BATCH = 8 if DEVICE == "cpu" else int(max(4, min(64, VRAM_GB * 2.2 * mult * (640 / IMGSZ) ** 2)))
print(f"batch {BATCH} | workers {WORKERS} | device {DEVICE}")

## 2. Model and pretrained weights
The COCO weights already know edges, textures and shapes; we replace the 80-class head by two classes (`smoke`=0, `fire`=1) and train. The weights file is downloaded from the Ultralytics release on first use.

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL_WEIGHTS)                       # downloads the pretrained weights if absent
print("weights file:", MODEL_WEIGHTS, "| exists here:", pathlib.Path(MODEL_WEIGHTS).exists())
model.info()                                      # layers, parameters, GFLOPs at 640 px
del model

## 3. Data and the leak-free split
`split_v2.csv` groups near-duplicate frames and keeps each group in one split (`ai-data/tools/dedup_groups.py`, RES-CAM-3). If the file is missing, generate it first (about 15 min on a slow disk):
`python ai-data/tools/dedup_groups.py features DATA_ROOT CACHE` (repeat until `done`), then `cluster ... --ham 6 --color 16`, then `split ... split_v2.csv --ham 6 --color 16`.

In [ ]:
import pandas as pd
if not SPLIT_CSV.exists():
    raise FileNotFoundError(f"{SPLIT_CSV} not found: build it with ai-data/tools/dedup_groups.py (see the cell above).")
df = pd.read_csv(SPLIT_CSV)
print(len(df), "images in the split file")
print(pd.crosstab(df["new_split"], df["class"]))

# integrity checks (stop on a real problem, warn on the rest)
img_path = lambda r: FIRE_ROOT / "data" / r.orig_split / "images" / r.image
lab_path = lambda r: FIRE_ROOT / "data" / r.orig_split / "labels" / (r.image.rsplit(".", 1)[0] + ".txt")
miss_img = [r.image for r in df.itertuples() if not img_path(r).exists()]
miss_lab = [r.image for r in df.itertuples() if not lab_path(r).exists()]
print("missing images:", len(miss_img), "| missing label files:", len(miss_lab))
assert not miss_img and not miss_lab, f"data incomplete, first missing: {(miss_img or miss_lab)[:3]}"
g = df.groupby("group")["new_split"].nunique()
assert (g == 1).all(), "a near-duplicate group is split across train/val/test: the split file is not leak-free"
assert df["image"].is_unique
cls_ids = set()
for r in df.sample(min(len(df), 2000), random_state=0).itertuples():
    for line in open(lab_path(r)):
        if line.strip(): cls_ids.add(int(line.split()[0]))
assert cls_ids <= {0, 1}, f"unexpected class ids {cls_ids}"
print("OK: files present, every group in one split, class ids", sorted(cls_ids))

## 4. Image processing
- **Lists** for Ultralytics (nothing is copied; labels are found by replacing `/images/` by `/labels/`).
- **Degraded copies** of the training images: resolution loss, blur, white balance, vignetting, haze, dirt, sensor noise, JPEG (OV2640-like, *assumed* ranges).
- **Night-like and sun-glare negatives**, made only from images without fire: in D-Fire most fire-only images are dark, so a network could learn "dark = fire".
- **Camera stamp** (`CameraNNNN`, date) masked or pasted with the same probability for every class, so it cannot predict the label.
Geometry never changes, so the boxes stay valid.

In [ ]:
LISTS = TRAIN / "lists"; DEG = TRAIN / ("deg_smoke" if PRESET == "smoke" else "deg")
import make_lists
make_lists.main(str(FIRE_ROOT), str(SPLIT_CSV), str(LISTS))

deg_list = DEG / "train_deg.txt"
if deg_list.exists() and SKIP_IF_DONE and PRESET != "smoke":
    print("degraded copies already built:", sum(1 for _ in open(deg_list)), "images (delete the deg folder to rebuild)")
else:
    cmd = [sys.executable, str(TRAIN / "build_degraded.py"), str(FIRE_ROOT), str(SPLIT_CSV), str(DEG), "--severity", "0.6"]
    if P["deg_limit"]: cmd += ["--limit", str(P["deg_limit"])]
    t0 = time.time(); subprocess.run(cmd, check=True); print(f"done in {time.time()-t0:.0f} s")

In [ ]:
# What the processing does, on real images (left: original, right: processed)
import matplotlib.pyplot as plt, cv2
from degrade import degrade, flare, night
from overlay import mask_overlay, paste_overlay
rng = np.random.default_rng(1)
tr = df[df.new_split == "train"]
rows = []
for c in ("both", "smoke", "background", "background"):
    r = tr[tr["class"] == c].sample(1, random_state=int(rng.integers(0, 10**6))).iloc[0]
    im = cv2.imread(str(img_path(r)))[..., ::-1]
    im = cv2.resize(im, (480, 270))
    if c == "background" and len(rows) == 2: out, name = night(im, rng), "night-like negative"
    elif c == "background": out, name = flare(im, rng, 0.9), "sun-glare negative"
    else: out, name = degrade(paste_overlay(mask_overlay(im), rng) if rng.random() < .5 else im, rng, 0.8)[0], "OV2640-like + stamp randomised"
    rows.append((im, out, f"{c}: {name}"))
fig, ax = plt.subplots(len(rows), 2, figsize=(10, 2.9 * len(rows)))
for i, (a, b, t) in enumerate(rows):
    ax[i, 0].imshow(a); ax[i, 1].imshow(b); ax[i, 1].set_title(t, fontsize=9)
    ax[i, 0].axis("off"); ax[i, 1].axis("off")
plt.tight_layout(); plt.show()

## 5. Fine-tuning (GPU)
Two phases, run by `train.py` in a separate process (this avoids the Windows multiprocessing problems of notebooks):
1. **Phase 1**: backbone frozen (`freeze=10`), the head learns the two classes.
2. **Phase 2**: all layers, lower learning rate, cosine schedule, early stopping.

Online augmentation: mosaic, scale, translation, horizontal flip, light HSV (the hue shift is kept tiny because fire is orange), mixup 0.1. The validation split is used for model selection; **the test split is never used for training or tuning**.

In [ ]:
BEST_PT = OUT_DIR / "phase2/weights/best.pt"
ONNX = OUT_DIR / "fire_smoke_yolo.onnx"
if BEST_PT.exists() and ONNX.exists() and SKIP_IF_DONE and PRESET != "smoke":
    print("trained weights found, training skipped:", BEST_PT)
else:
    cmd = [sys.executable, "-u", str(TRAIN / "train.py"), "--data", str(LISTS / "data_v2.yaml"), "--extra-train", str(deg_list),
           "--model", MODEL_WEIGHTS, "--out", str(OUT_DIR), "--imgsz", str(IMGSZ), "--batch", str(BATCH), "--device", DEVICE,
           "--workers", str(WORKERS), "--epochs1", str(P["epochs1"]), "--epochs2", str(P["epochs2"])]
    if P["smoke"]: cmd += ["--smoke"]
    print(" ".join(cmd), "\n")
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(TRAIN), bufsize=1)
    for line in proc.stdout:                       # live log (progress bars arrive as long lines: print the last part)
        print(line.rstrip()[-220:])
    rc = proc.wait()
    print(f"\nexit code {rc} after {(time.time()-t0)/60:.1f} min")
    assert rc == 0 and BEST_PT.exists() and ONNX.exists(), "training failed: read the log above"
print("weights:", BEST_PT, "|", round(BEST_PT.stat().st_size / 1e6, 1), "MB\nonnx   :", ONNX, "|", round(ONNX.stat().st_size / 1e6, 1), "MB")

In [ ]:
# Training curves (phase 2)
import matplotlib.pyplot as plt
res = pd.read_csv(OUT_DIR / "phase2/results.csv"); res.columns = [c.strip() for c in res.columns]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
for k in ("train/box_loss", "val/box_loss"):
    if k in res: ax[0].plot(res["epoch"], res[k], label=k)
ax[0].set_title("box loss"); ax[0].legend()
for k in ("train/cls_loss", "val/cls_loss"):
    if k in res: ax[1].plot(res["epoch"], res[k], label=k)
ax[1].set_title("class loss"); ax[1].legend()
for k in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
    if k in res: ax[2].plot(res["epoch"], res[k], label=k)
ax[2].set_title("validation mAP"); ax[2].legend()
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()
print("best validation mAP50:", round(float(res.get("metrics/mAP50(B)", pd.Series([np.nan])).max()), 3))

## 6. Decision metrics (the ones that matter)
Box mAP is secondary. The gateway needs one verdict per photo, so we score every image with **the same ONNX model and the same decoder as the gateway service**, then compute, at a fixed recall:
- the false-positive rate on images **without** fire or smoke and the expected **false alarms per day** (for `FRAMES_PER_DAY` photos),
- 95 % confidence intervals from a bootstrap **over near-duplicate groups** (correlated frames count once),
- slices: dark images, day images, images with and without the camera stamp.

The test list is scored twice: as is, and with the camera stamp masked. A clear loss on the stamp slice after masking means the model relied on it.

In [ ]:
import cv2
from tqdm.auto import tqdm
from gateway_vision.yolo import YoloOnnxClassifier
from overlay_probe import present, REG
clf_raw = YoloOnnxClassifier(str(ONNX), thr_fire=0.0, thr_smoke=0.0)
import onnxruntime as ort
print("ONNX runtime providers used:", clf_raw.sess.get_providers(), "(install onnxruntime-gpu to score faster on the GPU)")

def score_list(split, mask=False, limit=None):
    out_csv = OUT_DIR / f"scores_{split}_{'masked' if mask else 'plain'}.csv"
    if out_csv.exists() and SKIP_IF_DONE and PRESET != "smoke":
        return pd.read_csv(out_csv)
    paths = [l.strip() for l in open(LISTS / f"{split}.txt") if l.strip()]
    if limit: paths = paths[:limit]
    rows = []
    for p in tqdm(paths, desc=f"{split} {'masked' if mask else 'plain'}"):
        raw = pathlib.Path(p).read_bytes()
        img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
        g = img.mean(2)
        ov = int(present(g, REG["label"]) or present(g, REG["date"]))
        if mask:
            raw = cv2.imencode(".jpg", mask_overlay(img), [cv2.IMWRITE_JPEG_QUALITY, 95])[1].tobytes()
        dets, _ = clf_raw.detect(raw)
        rows.append((os.path.basename(p), max([d[1] for d in dets if d[0] == "smoke"], default=0.0),
                     max([d[1] for d in dets if d[0] == "fire"], default=0.0), ov))
    d = pd.DataFrame(rows, columns=["image", "score_smoke", "score_fire", "overlay"]); d.to_csv(out_csv, index=False)
    return d

L = P["eval_limit"]
S = {(sp, m): score_list(sp, m, L) for sp in ("val", "test") for m in (False, True) if not (sp == "val" and m)}
print({k: len(v) for k, v in S.items()})
for k, v in S.items():
    if max(v.score_fire.std(), v.score_smoke.std()) < 1e-3:
        print(f"WARNING {k}: the scores are (almost) constant -> the model collapsed or is untrained. "
              "Normal after PRESET='smoke' (3 images, 1 epoch); with 'quick' or 'full' it means a training problem: read the log of section 5.")

In [ ]:
import evaluate as ev
test_plain  = ev.load(OUT_DIR / "scores_test_plain.csv",  SPLIT_CSV)
test_masked = ev.load(OUT_DIR / "scores_test_masked.csv", SPLIT_CSV)
NB = 100 if PRESET == "smoke" else 500
print(f"=== test split, as is (recall target {RECALL_TARGET}, {FRAMES_PER_DAY} photos/day) ===")
r_plain = ev.run(test_plain, RECALL_TARGET, FRAMES_PER_DAY, NB)
print("\n=== test split, camera stamp masked ===")
r_mask = ev.run(test_masked, RECALL_TARGET, FRAMES_PER_DAY, NB, quiet=False)

In [ ]:
# Figures: ROC of the alarm decision, false alarms/day by slice, effect of masking the stamp
def curve(y, s):
    o = np.argsort(-s); y = y[o]; tp = np.cumsum(y); fp = np.cumsum(~y)
    return fp / max((~y).sum(), 1), tp / max(y.sum(), 1)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for d, nm, st in ((test_plain, "as is", "-"), (test_masked, "stamp masked", "--")):
    y = d["cls"] != "background"; s = np.maximum(d["ss"], d["sf"])
    x, t = curve(y, s); ax[0].plot(x, t, st, label=nm)
ax[0].set_xscale("log"); ax[0].set_xlim(1e-3, 1); ax[0].set_xlabel("false-positive rate (no-fire images)"); ax[0].set_ylabel("recall")
ax[0].axhline(RECALL_TARGET, color="grey", lw=.7); ax[0].set_title("alarm decision, test"); ax[0].legend()
names = [r["name"].replace("alarm ", "") for r in r_plain if r["name"].startswith("alarm")]
vals = [r for r in r_plain if r["name"].startswith("alarm")]
ax[1].bar(range(len(vals)), [v["fa_day"] for v in vals], yerr=[[v["fa_day"] - v["fa_lo"] for v in vals], [v["fa_hi"] - v["fa_day"] for v in vals]], capsize=3)
ax[1].set_xticks(range(len(vals))); ax[1].set_xticklabels(names, rotation=30, ha="right"); ax[1].set_ylabel(f"false alarms per day ({FRAMES_PER_DAY} photos)")
ax[1].set_title(f"at recall >= {RECALL_TARGET} (95 % CI over groups)")
ov_p = {r["name"]: r for r in r_plain}; ov_m = {r["name"]: r for r in r_mask}
keys = [k for k in ("alarm with overlay", "alarm no overlay", "alarm dark (<40)", "alarm day (>=40)") if k in ov_p and k in ov_m]
w = .38
ax[2].bar(np.arange(len(keys)) - w / 2, [ov_p[k]["auc"] for k in keys], w, label="as is")
ax[2].bar(np.arange(len(keys)) + w / 2, [ov_m[k]["auc"] for k in keys], w, label="stamp masked")
ax[2].set_xticks(range(len(keys))); ax[2].set_xticklabels([k.replace("alarm ", "") for k in keys], rotation=20, ha="right")
ax[2].set_ylim(0.5, 1.0); ax[2].set_ylabel("AUC"); ax[2].set_title("does masking the stamp hurt?"); ax[2].legend()
plt.tight_layout(); plt.show()

## 7. Thresholds and calibration (chosen on validation only)
- `thr_fire` and `thr_smoke`: the highest scores that still reach the recall target on the **validation** split. The test split then shows what they give on data not used for the choice.
- A **Platt calibration** (logistic fit of score → probability) is fitted on validation; the expected calibration error is reported on test before and after. Detector scores are not probabilities until this is done, and a calibrated score is still only as good as the validation data.

In [ ]:
from gateway_vision.evidence import PlattCalibrator, ece
val = ev.load(OUT_DIR / "scores_val_plain.csv", SPLIT_CSV)
isfire = lambda d: (d["cls"] == "fire") | (d["cls"] == "both")
issmoke = lambda d: (d["cls"] == "smoke") | (d["cls"] == "both")
def thr_at(s, y, rec):
    pos = np.sort(s[y])[::-1]
    return float(pos[int(math.ceil(rec * len(pos))) - 1])
thr_fire  = thr_at(val["sf"], isfire(val),  RECALL_TARGET)
thr_smoke = thr_at(val["ss"], issmoke(val), RECALL_TARGET)
print(f"chosen on validation: thr_fire={thr_fire:.3f}  thr_smoke={thr_smoke:.3f}")

def decision_table(d, tf, ts, label):
    cls = d["cls"]; nofire = cls == "background"
    alarm = (d["sf"] >= tf) | (d["ss"] >= ts)
    row = lambda nm, hit, base: dict(rule=label, what=nm, recall=float(hit[base].mean()) if base.sum() else np.nan)
    out = [row("fire images", d["sf"] >= tf, isfire(d)), row("smoke images", d["ss"] >= ts, issmoke(d)),
           row("any fire/smoke image (alarm)", alarm, ~nofire)]
    fpr = float(alarm[nofire].mean()); out.append(dict(rule=label, what=f"false alarms/day on no-fire scenes ({FRAMES_PER_DAY} photos)", recall=fpr * FRAMES_PER_DAY))
    return out
tab = pd.DataFrame(decision_table(test_plain, thr_fire, thr_smoke, "validation-chosen") + decision_table(test_plain, 0.25, 0.25, "default 0.25"))
tab["recall"] = tab["recall"].round(3)
display(tab) if "display" in globals() else print(tab)

In [ ]:
cal = {}
rows = []
for name, sv, yv, st, yt in (("fire", val["sf"], isfire(val), test_plain["sf"], isfire(test_plain)),
                              ("smoke", val["ss"], issmoke(val), test_plain["ss"], issmoke(test_plain))):
    c = PlattCalibrator().fit(sv, yv); cal[name] = dict(a=c.a, b=c.b, prevalence_val=c.prevalence)
    rows.append(dict(class_=name, ece_raw=ece(st, yt), ece_calibrated=ece(c.predict(st), yt), a=round(c.a, 3), b=round(c.b, 3)))
print(pd.DataFrame(rows).round(4).to_string(index=False))
THR_JSON = OUT_DIR / "thresholds.json"
json.dump(dict(thr_fire=thr_fire, thr_smoke=thr_smoke, recall_target=RECALL_TARGET, chosen_on="validation", calibration=cal), open(THR_JSON, "w"), indent=1)
print("written:", THR_JSON)

## 8. Export check: PyTorch model vs the ONNX the gateway uses
The gateway never runs PyTorch: it decodes the ONNX output itself. This cell compares both on 20 test images (largest fire and smoke score per image). Small differences come from the letterbox and the score threshold; a large one means an export or decoding problem.

In [ ]:
pt = YOLO(str(BEST_PT))
paths = [l.strip() for l in open(LISTS / "test.txt") if l.strip()]
random.Random(0).shuffle(paths); paths = paths[:20]
diff = []
for p in paths:
    full = pathlib.Path(p)
    r = pt.predict(str(full), imgsz=IMGSZ, conf=0.05, iou=0.5, verbose=False, device=DEVICE)[0]
    ptf = max([float(c) for c, k in zip(r.boxes.conf, r.boxes.cls) if int(k) == 1], default=0.0)
    pts = max([float(c) for c, k in zip(r.boxes.conf, r.boxes.cls) if int(k) == 0], default=0.0)
    dets, _ = clf_raw.detect(full.read_bytes())
    onf = max([d[1] for d in dets if d[0] == "fire"], default=0.0); ons = max([d[1] for d in dets if d[0] == "smoke"], default=0.0)
    diff.append((abs(ptf - onf), abs(pts - ons)))
diff = np.array(diff)
print(f"mean |PyTorch - ONNX| score: fire {diff[:,0].mean():.3f}, smoke {diff[:,1].mean():.3f} | worst {diff.max():.3f}")
print("OK" if diff.mean() < 0.1 else "CHECK: the two paths disagree more than expected (look at the letterbox / export settings)")

In [ ]:
# Publish the model for the gateway (models/) with a small card
MODEL_OUT = MODELS_DIR / "fire_smoke_yolo.onnx"
shutil.copyfile(ONNX, MODEL_OUT); shutil.copyfile(THR_JSON, MODELS_DIR / "thresholds.json")
card = dict(file=MODEL_OUT.name, weights=MODEL_WEIGHTS, imgsz=IMGSZ, preset=PRESET, epochs=[P["epochs1"], P["epochs2"]], gpu=GPU_NAME,
            trained_utc=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), split=str(SPLIT_CSV.name), classes={0: "smoke", 1: "fire"},
            thr_fire=thr_fire, thr_smoke=thr_smoke, recall_target=RECALL_TARGET,
            test_alarm=next((r for r in r_plain if r["name"] == "alarm (any)"), None),
            notes="D-Fire only unless real OV2640 photos were added; degradations are assumptions; scores are not calibrated probabilities")
json.dump(card, open(MODELS_DIR / "model_card.json", "w"), indent=1, default=float)
print("published:", MODEL_OUT, round(MODEL_OUT.stat().st_size / 1e6, 1), "MB")

## 9. The gateway uses the model: send photos, get verdicts
This starts the real gateway HTTP service with the model and the thresholds, then sends photos **exactly like a box does** (`POST /v1/frame`, headers `X-Box-Id`, `X-Req-Id`, `X-Time`, `X-Reasons`). The service answers `{"v":1,"id":..,"label":"fire|smoke|background","conf":..,"ms":..}`. With `?debug=1` it also returns the detected boxes (for display only; the box never asks for them).

Photos sent: a few test images per class, plus anything in `ai-data/data/camera_photos/` (put real OV2640 photos there to see how the model behaves on them).

In [ ]:
import threading
from gateway_vision.server import make_server, build_classifier
from gateway_vision.client import send_photo, draw_boxes, IMAGES
clf = build_classifier(str(MODEL_OUT), "yolo", thresholds_json=str(MODELS_DIR / "thresholds.json"))
srv = make_server(clf, "127.0.0.1", 0, store=OUT_DIR / "received")
threading.Thread(target=srv.serve_forever, daemon=True).start()
URL = f"http://127.0.0.1:{srv.server_port}"
print("gateway service:", URL, "| model", clf.name, f"| thr_fire {clf.thr_fire:.3f} thr_smoke {clf.thr_smoke:.3f}")

# photos to send: 3 per class from the test split + user photos
te = df[df.new_split == "test"]
photos = []
for c in ("fire", "smoke", "both", "background"):
    for r in te[te["class"] == c].sample(min(3, int((te["class"] == c).sum())), random_state=1).itertuples():
        photos.append((img_path(r), c))
if MY_PHOTOS.exists():
    photos += [(p, "?") for p in sorted(MY_PHOTOS.iterdir()) if p.suffix.lower() in IMAGES]
print(len(photos), "photos to send")
results = []
for i, (p, truth) in enumerate(photos, 1):
    jpeg = pathlib.Path(p).read_bytes()
    r = send_photo(URL, jpeg, box_id=1, req_id=i, reasons=0, debug=True)
    a = r["answer"] or {}
    results.append(dict(photo=pathlib.Path(p).name, truth=truth, status=r["status"], verdict=a.get("label", a.get("error")), conf=a.get("conf"),
                        model_ms=a.get("ms"), round_trip_ms=r["round_trip_ms"], boxes=a.get("boxes", []), path=str(p)))
rd = pd.DataFrame(results)
print(rd.drop(columns=["boxes", "path"]).to_string(index=False))
print(f"\nmedian model time {rd.model_ms.median():.0f} ms | median round trip {rd.round_trip_ms.median():.0f} ms  (one photo per 30 s; the box waits for this answer with its Wi-Fi on)")

In [ ]:
# The verdicts on the photos, with the boxes the service found
n = len(rd); cols = 4; rows_n = math.ceil(n / cols)
fig, axs = plt.subplots(rows_n, cols, figsize=(4.4 * cols, 2.7 * rows_n)); axs = np.atleast_2d(axs)
for ax in axs.ravel(): ax.axis("off")
for ax, r in zip(axs.ravel(), rd.itertuples()):
    tmp = OUT_DIR / "annotated" / r.photo; tmp.parent.mkdir(exist_ok=True)
    draw_boxes(r.path, r.boxes, tmp)
    ax.imshow(plt.imread(tmp)); ax.set_title(f"truth {r.truth} -> {r.verdict} ({r.conf})", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
srv.shutdown()
print("service stopped. To run it for real (the box or the dashboard sends the photos):\n")
print(f'  cd {VISION / "python"}')
print(f'  python -m gateway_vision.server --model "{MODEL_OUT}" --thresholds "{MODELS_DIR / "thresholds.json"}" --port 8080 --store received\n')
print("and to send photos by hand from any machine:\n")
print('  python -m gateway_vision.client http://GATEWAY_IP:8080 --folder my_photos --annotate annotated --csv results.csv')

## What this does not prove
- **No real camera image was used** unless you put OV2640 photos in `ai-data/data/camera_photos/` and trained or tested on them. D-Fire frames come from fixed surveillance cameras.
- Degradation ranges, night-like and glare negatives are **assumptions**, not a model of the OV2640.
- The split removes near-duplicates found by a perceptual hash; **frames from the same site at another time may still look alike** (no site id in the data), so the test numbers can remain optimistic.
- Fire-only test images are few (about 170): the confidence intervals on the fire class are wide.
- Repeated photos of one scene are correlated: accumulating them (SPRT) does not cut false alarms as much as independence would suggest (RES-CAM-5).
- Log this run in `docs/research/box-camera.md` (question, method, result, reading, decision), failures included.